# Option A Bonus Calibration (Full)

Sweeps every bonus and penalty parameter under the Option A post-sigmoid architecture.

| Component | Formula | Positions |
|---|---|---|
| Goal bonus | `α × log2(goals+1) × isolation` | ST, Winger, CM, CDM, CB, FB |
| Assist bonus | `γ × log2(assists+1) × isolation` | ST, Winger, CM, CDM, CB, FB |
| Mastery | `min(excess, excess_cap) × w × impact` (proportional, capped) | All |
| CDM Reliable Pivot | Flat, tiered by possession_lost | CDM |
| ST Hold-Up Bonus | Proportional, capped | ST |
| ST Black Hole Penalty | Proportional, capped | ST |
| ST Wasteful Finisher Penalty | Proportional, capped | ST |
| Winger Wastefulness Penalty | Per excess shot | Winger |
| CB/FB Clean Sheet | xG-tiered × linear ramp | CB, FB |
| CDM/CM Clean Sheet | Flat × linear ramp | CDM, CM |
| CB/FB Collapse Penalty | Flat (−), 60+ mins, 3+ goals conceded | CB, FB |

All sweeps use pandas prediction — no service re-runs after cell 3.

In [1]:
from pathlib import Path
import sys, json, math
import pandas as pd
import numpy as np

project_root = Path("..").resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.services.analytics.match_ratings_service import MatchRatingsService

TEAM_NAME    = "Valencia CF"
MATCHES_PATH = project_root / "tests" / "fixtures" / "testing_data" / "valencia_cf_1" / "matches.json"

with open(project_root / "config" / "performance_weights.json")    as f: weights    = json.load(f)
with open(project_root / "config" / "performance_means_stds.json") as f: means_stds = json.load(f)
with open(MATCHES_PATH) as f: data = json.load(f)

POSITION_GROUP_MAP = {
    'ST': 'ST', 'LW': 'Winger', 'RW': 'Winger',
    'CM': 'CM', 'CDM': 'CDM', 'CB': 'CB', 'LB': 'FB', 'RB': 'FB',
}
GROUPS = ['ST', 'Winger', 'CM', 'CDM', 'CB', 'FB']
XG_PER_SHOT = 0.1116

print(f"Loaded {len(data)} matches")

Loaded 155 matches


In [2]:
# ── Capture service ───────────────────────────────────────────────────────────
# Calls super() first so z_score floors are applied in-place before we capture
# them — mastery conditions must use floored z_scores to match production.
# base_rating is then computed from the correct floored z_scores.

class BonusCaptureService(MatchRatingsService):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._captures = []
        self._cur_mastery_fires = []

    def _apply_mastery_bonus(self, raw_score, z_scores, key_a, key_b, threshold, weight):
        """Record mastery fire data without applying to raw_score."""
        mastery = min(z_scores.get(key_a, 0.0), z_scores.get(key_b, 0.0))
        if mastery > threshold:
            self._cur_mastery_fires.append({
                'key_a': key_a, 'key_b': key_b,
                'threshold': threshold, 'excess': mastery - threshold,
            })
        return raw_score  # do not apply

    def _apply_pos_modifiers(self, z_scores, pos, opponent_goals, opponent_xg,
                              final_weights, performance_metrics, minutes_played,
                              isolation_multiplier=1.0):
        self._cur_mastery_fires = []

        # Call super() first — applies z_score floors in-place AND fires mastery hooks
        super()._apply_pos_modifiers(
            z_scores=z_scores, pos=pos, opponent_goals=opponent_goals,
            opponent_xg=opponent_xg, final_weights=final_weights,
            performance_metrics=performance_metrics, minutes_played=minutes_played,
            isolation_multiplier=isolation_multiplier)

        # z_scores now has floors applied — compute base_rating from correct values
        dot    = self._calculate_dot_product(z_scores=z_scores, weights=final_weights)
        impact = math.sqrt(min(minutes_played, 90.0) / 90.0)
        base_rating = self._apply_sigmoid_transformation(raw_score=dot * impact)

        self._captures.append({
            'pos':     pos,
            'group':   POSITION_GROUP_MAP.get(pos, pos),
            'minutes': minutes_played,
            'impact':  impact,
            'isolation': isolation_multiplier,
            'base_rating': base_rating,
            'opponent_goals': opponent_goals,
            'opponent_xg':   opponent_xg,
            'goals':   performance_metrics.get('goals',   0),
            'assists': performance_metrics.get('assists', 0),
            # z_scores (post-floor)
            'tackles_z':           z_scores.get('tackles_p90_z', 0.0),
            'possession_won_z':    z_scores.get('possession_won_p90_z', 0.0),
            'passes_z':            z_scores.get('passes_p90_z', 0.0),
            'dribbles_z':          z_scores.get('dribbles_p90_z', 0.0),
            'xt_bonus_z':          z_scores.get('xt_bonus_p90_z', 0.0),
            'distance_sprinted_z': z_scores.get('distance_sprinted_p90_z', 0.0),
            # raw metrics
            'pass_accuracy_raw':   performance_metrics.get('pass_accuracy', 0.0),
            'possession_lost_raw': performance_metrics.get('possession_lost', 0.0),
            'passes_raw':          performance_metrics.get('passes', 0.0),
            'dribbles_raw':        performance_metrics.get('dribbles', 0.0),
            'shots_raw':           performance_metrics.get('shots', 0.0),
            'mastery_fires': list(self._cur_mastery_fires),
        })
        return 0.0, 0.0

print("BonusCaptureService defined.")

BonusCaptureService defined.


In [3]:
svc = BonusCaptureService(weights, means_stds)
for match in data:
    mo = match['data']; hl = mo['half_length']
    for perf in match['player_performances']:
        if perf['performance_type'] != 'Outfield': continue
        svc.calculate_outfield_rating(perf, mo, hl, TEAM_NAME)

df = pd.DataFrame(svc._captures)
df['scored']      = df['goals']   >= 1
df['assisted']    = df['assists'] >= 1
df['clean_sheet'] = df['opponent_goals'] == 0
df['n_mastery']   = df['mastery_fires'].apply(len)
sub = df[df['group'].isin(GROUPS)]

print(f"Captured {len(df)} outfield appearances")
print()

MASTERY_CONDITIONS = {
    'CB':     [('tackles_z','possession_won_z',1.5,'Dominant Stopper'),
               ('passes_z','possession_won_z',1.0,'Ball Playing Def')],
    'FB':     [('tackles_z','possession_won_z',1.0,'Third CB'),
               ('distance_sprinted_z','xt_bonus_z',1.0,'Express Train'),
               ('passes_z','dribbles_z',1.0,'Wide Playmaker')],
    'CDM':    [('tackles_z','possession_won_z',1.5,'Destroyer'),
               ('passes_z','dribbles_z',1.5,'Deep-Lying PM')],
    'CM':     [('tackles_z','possession_won_z',1.5,'Enforcer'),
               ('passes_z','dribbles_z',1.2,'Progression Eng')],
    'Winger': [('dribbles_z','xt_bonus_z',1.5,'Direct Threat'),
               ('passes_z','xt_bonus_z',1.5,'Wide Playmaker'),
               ('tackles_z','possession_won_z',1.0,'Pressing Fwd')],
    'ST':     [('passes_z','dribbles_z',1.5,'Complete Fwd')],
}

print("Mastery condition trigger rates:")
print(f"{'Group':<8} {'Condition':<20} {'Rate':>6} {'Mean excess':>12} {'Max excess':>11}")
print("-" * 62)
for g, conditions in MASTERY_CONDITIONS.items():
    grp = sub[sub['group'] == g]
    for za, zb, thresh, name in conditions:
        min_z = grp[[za, zb]].min(axis=1)
        excess = (min_z - thresh).clip(lower=0)
        fired = excess > 0
        rate = fired.mean()
        mean_ex = excess[fired].mean() if fired.any() else 0
        max_ex  = excess.max()
        print(f"{g:<8} {name:<20} {rate:>6.1%} {mean_ex:>12.3f} {max_ex:>11.3f}")

print()
print("CDM Reliable Pivot trigger rates (60+ mins, pass_acc>=92, passes_z>1.0):")
cdm_all = sub[sub['group']=='CDM']
eligible = cdm_all[(cdm_all['minutes']>=60) & (cdm_all['pass_accuracy_raw']>=92.0) & (cdm_all['passes_z']>1.0)]
pm = eligible[eligible['possession_lost_raw']==0]
rs = eligible[(eligible['possession_lost_raw']>0) & (eligible['possession_lost_raw']<=1)]
print(f"  Eligible (gate conditions met): {len(eligible):3d} / {len(cdm_all)} ({len(eligible)/max(len(cdm_all),1):.1%})")
print(f"  Perfect Metronome (poss_lost==0): {len(pm):3d} / {len(cdm_all)} ({len(pm)/max(len(cdm_all),1):.1%})")
print(f"  Reliable Shift    (poss_lost<=1): {len(rs):3d} / {len(cdm_all)} ({len(rs)/max(len(cdm_all),1):.1%})")

print()
print("ST special condition trigger rates:")
st = sub[sub['group']=='ST'].copy()
st['pos_inv']  = st['passes_raw'] + st['dribbles_raw'] + st['shots_raw']
st['safe_loss']= st['possession_lost_raw'].clip(lower=1)
st['safe_inv'] = st['pos_inv'].clip(lower=1)
holdup = st[(st['pos_inv']>=15) & ((st['pos_inv']/st['safe_loss'])>4.0)]
bh     = st[(st['possession_lost_raw']>4) & ((st['possession_lost_raw']/st['safe_inv'])>1.5)]
wf_deficit = st['shots_raw']*XG_PER_SHOT - st['goals']
wf = st[(st['shots_raw']>3) & (wf_deficit>0.75)]
print(f"  Hold-Up Bonus trigger:      {len(holdup):3d} / {len(st)} ({len(holdup)/len(st):.1%})")
print(f"  Black Hole Penalty trigger: {len(bh):3d} / {len(st)} ({len(bh)/len(st):.1%})")
print(f"  Wasteful Finisher trigger:  {len(wf):3d} / {len(st)} ({len(wf)/len(st):.1%})")

print()
print("CB/FB clean sheet xG distribution:")
for g in ['CB','FB']:
    cs_rows = sub[(sub['group']==g) & sub['clean_sheet']]
    n = max(len(cs_rows), 1)
    low  = (cs_rows['opponent_xg'] <= 1.0).sum()
    mid  = ((cs_rows['opponent_xg'] > 1.0) & (cs_rows['opponent_xg'] < 2.0)).sum()
    high = (cs_rows['opponent_xg'] >= 2.0).sum()
    print(f"  {g}: {len(cs_rows)} clean sheets — "
          f"xG<=1.0: {low} ({low/n:.0%})  1.0-2.0: {mid} ({mid/n:.0%})  xG>=2.0: {high} ({high/n:.0%})")

Captured 2267 outfield appearances

Mastery condition trigger rates:
Group    Condition              Rate  Mean excess  Max excess
--------------------------------------------------------------
CB       Dominant Stopper       1.4%        0.252       0.598
CB       Ball Playing Def       0.2%        0.028       0.028
FB       Third CB               5.6%        0.374       0.890
FB       Express Train          7.5%        0.288       0.768
FB       Wide Playmaker         2.7%        0.587       2.325
CDM      Destroyer              0.4%        0.307       0.307
CDM      Deep-Lying PM          2.2%        0.411       0.864
CM       Enforcer               1.1%        0.281       0.590
CM       Progression Eng        5.7%        0.407       1.435
Winger   Direct Threat          0.9%        0.355       0.475
Winger   Wide Playmaker         0.7%        0.285       0.399
Winger   Pressing Fwd           4.8%        0.553       1.540
ST       Complete Fwd           7.2%        1.457       3.449


In [4]:
# ── Threshold diagnostic cell — tweak and re-run freely ──────────────────────
# All computations use captured z_scores/raw metrics — no service re-run needed.

# ── Mastery thresholds (lower = fires more often) ─────────────────────────────
MASTERY_THRESHOLDS = {
    'CB': {
        'Dominant Stopper':  ('tackles_z', 'possession_won_z', 1.2),
        'Ball Playing Def':  ('passes_z',  'dribbles_z', 1.0),
    },
    'FB': {
        'Third CB':          ('tackles_z',           'possession_won_z',    1.0),
        'Express Train':     ('distance_sprinted_z', 'xt_bonus_z',          1.0),
        'Wide Playmaker':    ('passes_z',             'dribbles_z',          1.0),
    },
    'CDM': {
        'Destroyer':         ('tackles_z', 'possession_won_z', 1.2),
        'Deep-Lying PM':     ('passes_z',  'dribbles_z',       1.2),
    },
    'CM': {
        'Enforcer':          ('tackles_z', 'possession_won_z', 1.2),
        'Progression Eng':   ('passes_z',  'dribbles_z',       1.2),
    },
    'Winger': {
        'Direct Threat':     ('dribbles_z', 'xt_bonus_z',          1.2),
        'Wide Playmaker':    ('passes_z',   'xt_bonus_z',           1.0),
        'Pressing Fwd':      ('tackles_z',  'possession_won_z',     1.0),
    },
    'ST': {
        'Complete Fwd':      ('passes_z',  'dribbles_z',        1.5),
    },
}

# ── CDM Reliable Pivot gate conditions ────────────────────────────────────────
CDM_PIVOT_MIN_MINUTES  = 45
CDM_PIVOT_MIN_PASS_ACC = 88.0   # was 92.0 — try relaxing
CDM_PIVOT_MIN_PASSES_Z = 0.8
CDM_PIVOT_PM_THRESHOLD = 0      # possession_lost == this for Perfect Metronome
CDM_PIVOT_RS_THRESHOLD = 2      # possession_lost <= this for Reliable Shift (was 1)

# ── ST Hold-Up Bonus conditions ───────────────────────────────────────────────
ST_HOLDUP_MIN_POS_INV       = 25    # was 15 — tighten to reduce 55% trigger rate
ST_HOLDUP_MIN_RETENTION     = 7.0   # was 4.0

# ── ST Black Hole Penalty conditions ─────────────────────────────────────────
ST_BLACKHOLE_MIN_POSS_LOST  = 2     # was 4 — lower to make it fire
ST_BLACKHOLE_MIN_TURNOVER   = 1.0   # was 1.5

# ── ST Wasteful Finisher conditions ───────────────────────────────────────────
ST_WASTEFUL_MIN_SHOTS       = 3     # was 3 — lower to make it fire
ST_WASTEFUL_MIN_DEFICIT     = 0.40  # was 0.75

# ═════════════════════════════════════════════════════════════════════════════
# Diagnostics — nothing below this line needs editing
# ═════════════════════════════════════════════════════════════════════════════

print("Mastery condition trigger rates:")
print(f"{'Group':<8} {'Condition':<20} {'Rate':>6} {'Mean excess':>12} {'Max excess':>11}")
print("-" * 62)
for g, conditions in MASTERY_THRESHOLDS.items():
    grp = sub[sub['group'] == g]
    for name, (za, zb, thresh) in conditions.items():
        min_z  = grp[[za, zb]].min(axis=1)
        excess = (min_z - thresh).clip(lower=0)
        fired  = excess > 0
        rate    = fired.mean()
        mean_ex = excess[fired].mean() if fired.any() else 0
        max_ex  = excess.max()
        flag = "  *** too high" if rate > 0.15 else ("  * low" if rate < 0.01 else "")
        print(f"{g:<8} {name:<20} {rate:>6.1%} {mean_ex:>12.3f} {max_ex:>11.3f}{flag}")

print()
print("CDM Reliable Pivot trigger rates:")
cdm_all  = sub[sub['group'] == 'CDM']
eligible = cdm_all[
    (cdm_all['minutes']          >= CDM_PIVOT_MIN_MINUTES) &
    (cdm_all['pass_accuracy_raw']>= CDM_PIVOT_MIN_PASS_ACC) &
    (cdm_all['passes_z']         >  CDM_PIVOT_MIN_PASSES_Z)
]
pm = eligible[eligible['possession_lost_raw'] == CDM_PIVOT_PM_THRESHOLD]
rs = eligible[
    (eligible['possession_lost_raw'] >  CDM_PIVOT_PM_THRESHOLD) &
    (eligible['possession_lost_raw'] <= CDM_PIVOT_RS_THRESHOLD)
]
n = max(len(cdm_all), 1)
print(f"  Gate (pass_acc>={CDM_PIVOT_MIN_PASS_ACC}, passes_z>{CDM_PIVOT_MIN_PASSES_Z}, {CDM_PIVOT_MIN_MINUTES}+min): "
      f"{len(eligible):3d} / {len(cdm_all)} ({len(eligible)/n:.1%})")
print(f"  Perfect Metronome (poss_lost=={CDM_PIVOT_PM_THRESHOLD}):    "
      f"{len(pm):3d} / {len(cdm_all)} ({len(pm)/n:.1%})")
print(f"  Reliable Shift    (poss_lost<={CDM_PIVOT_RS_THRESHOLD}):    "
      f"{len(rs):3d} / {len(cdm_all)} ({len(rs)/n:.1%})")

print()
print("ST special condition trigger rates:")
st = sub[sub['group'] == 'ST'].copy()
st['pos_inv']  = st['passes_raw'] + st['dribbles_raw'] + st['shots_raw']
st['safe_loss']= st['possession_lost_raw'].clip(lower=1)
st['safe_inv'] = st['pos_inv'].clip(lower=1)

holdup = st[
    (st['pos_inv']   >= ST_HOLDUP_MIN_POS_INV) &
    ((st['pos_inv'] / st['safe_loss']) > ST_HOLDUP_MIN_RETENTION)
]
bh = st[
    (st['possession_lost_raw'] >  ST_BLACKHOLE_MIN_POSS_LOST) &
    ((st['possession_lost_raw'] / st['safe_inv']) > ST_BLACKHOLE_MIN_TURNOVER)
]
wf_deficit = st['shots_raw'] * XG_PER_SHOT - st['goals']
wf = st[
    (st['shots_raw'] > ST_WASTEFUL_MIN_SHOTS) &
    (wf_deficit      > ST_WASTEFUL_MIN_DEFICIT)
]
n_st = len(st)
print(f"  Hold-Up Bonus  (inv>={ST_HOLDUP_MIN_POS_INV}, ratio>{ST_HOLDUP_MIN_RETENTION}):    "
      f"{len(holdup):3d} / {n_st} ({len(holdup)/n_st:.1%})")
print(f"  Black Hole     (poss_lost>{ST_BLACKHOLE_MIN_POSS_LOST}, ratio>{ST_BLACKHOLE_MIN_TURNOVER}): "
      f"{len(bh):3d} / {n_st} ({len(bh)/n_st:.1%})")
print(f"  Wasteful Fin.  (shots>{ST_WASTEFUL_MIN_SHOTS}, deficit>{ST_WASTEFUL_MIN_DEFICIT}):      "
      f"{len(wf):3d} / {n_st} ({len(wf)/n_st:.1%})")

Mastery condition trigger rates:
Group    Condition              Rate  Mean excess  Max excess
--------------------------------------------------------------
CB       Dominant Stopper       4.2%        0.279       0.898
CB       Ball Playing Def       1.4%        0.540       1.447
FB       Third CB               5.6%        0.374       0.890
FB       Express Train          7.5%        0.288       0.768
FB       Wide Playmaker         2.7%        0.587       2.325
CDM      Destroyer              4.0%        0.189       0.607
CDM      Deep-Lying PM          4.0%        0.459       1.164
CM       Enforcer               2.5%        0.290       0.890
CM       Progression Eng        5.7%        0.407       1.435
Winger   Direct Threat          1.3%        0.511       0.775
Winger   Wide Playmaker         1.5%        0.436       0.899
Winger   Pressing Fwd           4.8%        0.553       1.540
ST       Complete Fwd           7.2%        1.457       3.449

CDM Reliable Pivot trigger rates:
 

In [5]:
import copy

def _mastery(z_a, z_b, threshold, weight, excess_cap, impact):
    excess = min(z_a, z_b) - threshold
    if excess > 0:
        return min(excess, excess_cap) * weight * impact
    return 0.0


def predict_rating(row, cfg: dict) -> float:
    g   = row['group']
    rat = row['base_rating']
    imp = row['impact']
    iso = row['isolation']
    mp  = row['minutes']
    mw  = cfg['mastery_weight']
    mc  = cfg['mastery_excess_cap']

    # z_score lookup — keyed to match MASTERY_THRESHOLDS column names
    z = {
        'tackles_z':           row['tackles_z'],
        'possession_won_z':    row['possession_won_z'],
        'passes_z':            row['passes_z'],
        'dribbles_z':          row['dribbles_z'],
        'xt_bonus_z':          row['xt_bonus_z'],
        'distance_sprinted_z': row['distance_sprinted_z'],
    }

    # Goal bonus
    if row['goals'] >= 1:
        rat += cfg['alpha'].get(g, 0.0) * math.log2(row['goals'] + 1) * iso

    # Assist bonus
    if row['assists'] >= 1:
        rat += cfg['gamma'].get(g, 0.0) * math.log2(row['assists'] + 1) * iso

    # Mastery — data-driven from MASTERY_THRESHOLDS so diagnostic cell changes
    # automatically flow through here without further edits.
    if g in MASTERY_THRESHOLDS:
        for name, (za, zb, thresh) in MASTERY_THRESHOLDS[g].items():
            rat += _mastery(z[za], z[zb], thresh, mw, mc, imp)

    # CDM Reliable Pivot — uses CDM_PIVOT_* variables from diagnostic cell
    if (g == 'CDM'
            and mp >= CDM_PIVOT_MIN_MINUTES
            and row['pass_accuracy_raw'] >= CDM_PIVOT_MIN_PASS_ACC
            and z['passes_z'] > CDM_PIVOT_MIN_PASSES_Z):
        pl = row['possession_lost_raw']
        if pl == CDM_PIVOT_PM_THRESHOLD:
            rat += cfg['cdm_perfect_metronome']
        elif pl <= CDM_PIVOT_RS_THRESHOLD:
            rat += cfg['cdm_reliable_shift']

    # ST penalties and hold-up — uses ST_* variables from diagnostic cell
    if g == 'ST':
        pos_inv   = row['passes_raw'] + row['dribbles_raw'] + row['shots_raw']
        safe_loss = max(row['possession_lost_raw'], 1.0)
        safe_inv  = max(pos_inv, 1.0)
        # Hold-Up Bonus
        if pos_inv >= ST_HOLDUP_MIN_POS_INV and (pos_inv / safe_loss) > ST_HOLDUP_MIN_RETENTION:
            excess_ret = max(0, pos_inv - row['possession_lost_raw'] * 3.0)
            rat += min(excess_ret * cfg['st_holdup_scale'], cfg['st_holdup_cap'])
        # Black Hole Penalty
        if (row['possession_lost_raw'] > ST_BLACKHOLE_MIN_POSS_LOST
                and (row['possession_lost_raw'] / safe_inv) > ST_BLACKHOLE_MIN_TURNOVER):
            excess_loss = max(0, row['possession_lost_raw'] - pos_inv)
            rat -= min(excess_loss * cfg['st_blackhole_scale'], cfg['st_blackhole_cap'])
        # Wasteful Finisher
        deficit = row['shots_raw'] * XG_PER_SHOT - row['goals']
        if row['shots_raw'] > ST_WASTEFUL_MIN_SHOTS and deficit > ST_WASTEFUL_MIN_DEFICIT:
            rat -= min(deficit * cfg['st_wasteful_scale'], cfg['st_wasteful_cap'])

    # Winger Wastefulness — fires when winger had 3+ wasted shots AND zero creative output
    if g == 'Winger':
        wasted   = row['shots_raw'] - row['goals']   # shots that didn't score
        creative = row['goals'] + row['assists']     # any positive end product
        if wasted >= 3 and creative == 0:
            rat -= (wasted - 2) * cfg['winger_wasteful_per_shot']

    # CB/FB xG-tiered clean sheet
    if g in ('CB', 'FB') and row['clean_sheet']:
        mult = cfg['cs_fb_ratio'] if g == 'FB' else 1.0
        ramp = (min(mp, 60.0) / 60.0) ** cfg['cs_ramp_n']
        xg   = row['opponent_xg']
        if xg <= 1.0:   rat += cfg['cs_cb_low_xg']  * mult * ramp
        elif xg < 2.0:  rat += cfg['cs_cb_mid_xg']  * mult * ramp
        else:           rat += cfg['cs_cb_high_xg'] * mult * ramp

    # CB/FB Collapse Penalty
    if g in ('CB', 'FB') and row['opponent_goals'] >= 3 and mp >= 60:
        rat -= cfg['collapse_penalty']

    # CDM/CM xG-tiered clean sheet (same CB tiers scaled by position ratio)
    if row['clean_sheet'] and g in ('CDM', 'CM'):
        ramp  = (min(mp, 60.0) / 60.0) ** cfg['cs_ramp_n']
        xg    = row['opponent_xg']
        ratio = cfg['cs_cdm_ratio'] if g == 'CDM' else cfg['cs_cm_ratio']
        if xg <= 1.0:   rat += cfg['cs_cb_low_xg']  * ratio * ramp
        elif xg < 2.0:  rat += cfg['cs_cb_mid_xg']  * ratio * ramp
        else:           rat += cfg['cs_cb_high_xg'] * ratio * ramp

    return round(max(0.0, min(10.0, rat)), 3)


def summary(df_sub, cfg):
    d = df_sub.copy()
    d['pred'] = d.apply(lambda r: predict_rating(r, cfg), axis=1)
    rows = []
    for g in GROUPS:
        grp = d[d['group'] == g]
        if grp.empty: continue
        sc  = grp[grp['scored']]
        nsc = grp[~grp['scored']]
        rows.append({
            'group': g, 'n': len(grp),
            'mean':  round(grp['pred'].mean(), 3),
            'p25':   round(grp['pred'].quantile(0.25), 3),
            'p50':   round(grp['pred'].quantile(0.50), 3),
            'p75':   round(grp['pred'].quantile(0.75), 3),
            'non_sc_mean': round(nsc['pred'].mean(), 3),
            'sc_mean': round(sc['pred'].mean(), 3) if len(sc) >= 3 else None,
            'sc_gap':  round(sc['pred'].mean() - nsc['pred'].mean(), 3) if len(sc) >= 3 else None,
        })
    return pd.DataFrame(rows).set_index('group')


CFG = {
    'alpha': {'ST': 0.30, 'Winger': 0.40, 'CM': 0.20, 'CDM': 0.10, 'CB': 0.15, 'FB': 0.12},
    'gamma': {'ST': 0.25, 'Winger': 0.25, 'CM': 0.25, 'CDM': 0.15, 'CB': 0.15, 'FB': 0.15},
    'mastery_weight': 0.10,     'mastery_excess_cap': 1.5,
    'cdm_perfect_metronome': 0.20,  'cdm_reliable_shift': 0.10,
    'st_holdup_scale': 0.010,   'st_holdup_cap': 0.25,
    'st_blackhole_scale': 0.040, 'st_blackhole_cap': 0.35,
    'st_wasteful_scale': 0.12,  'st_wasteful_cap': 0.40,
    'winger_wasteful_per_shot': 0.05,
    'cs_cb_low_xg': 0.30,  'cs_cb_mid_xg': 0.20,  'cs_cb_high_xg': 0.08,
    'cs_fb_ratio': 0.70,   'cs_cdm_ratio': 0.40,  'cs_cm_ratio': 0.28,
    'cs_ramp_n': 1.0,      'collapse_penalty': 0.20,
}
ZERO_CFG = {
    'alpha': {}, 'gamma': {},
    'mastery_weight': 0.0,  'mastery_excess_cap': 1.5,
    'cdm_perfect_metronome': 0.0,  'cdm_reliable_shift': 0.0,
    'st_holdup_scale': 0.0,   'st_holdup_cap': 0.0,
    'st_blackhole_scale': 0.0, 'st_blackhole_cap': 0.0,
    'st_wasteful_scale': 0.0,  'st_wasteful_cap': 0.0,
    'winger_wasteful_per_shot': 0.0,
    'cs_cb_low_xg': 0.0,  'cs_cb_mid_xg': 0.0,  'cs_cb_high_xg': 0.0,
    'cs_fb_ratio': 0.70,   'cs_cdm_ratio': 0.0,  'cs_cm_ratio': 0.0,
    'cs_ramp_n': 1.0,      'collapse_penalty': 0.0,
}

print("Baseline (no bonuses):")
print(summary(sub, ZERO_CFG).to_string())


Baseline (no bonuses):
          n   mean    p25    p50    p75  non_sc_mean  sc_mean  sc_gap
group                                                                
ST      264  6.348  5.808  6.065  6.700        6.024    6.817   0.794
Winger  459  6.149  5.769  5.967  6.414        6.073    6.390   0.317
CM      473  6.138  5.787  5.987  6.410        6.017    6.589   0.572
CDM     226  6.229  5.823  6.075  6.552        6.219    6.481   0.262
CB      432  6.186  5.821  6.028  6.540        6.176    6.630   0.454
FB      411  6.205  5.822  6.071  6.560        6.205      NaN     NaN


In [6]:
# ── Sweep 1: Goal bonus (α) — all other bonuses zeroed ───────────────────────

print("Goal bonus sweep (α only):")
print(f"{'α_ST':>6} {'α_Wng':>6} {'α_CM':>5} {'α_CDM':>6} {'α_CB':>5} {'α_FB':>5}  "
      f"{'ST_sc_gap':>10} {'Wng_sc_gap':>11} {'CM_sc_gap':>10}  "
      f"{'ST_mean':>8} {'Wng_mean':>9} {'CM_mean':>8} {'CB_mean':>8}")
print("-" * 110)

for ast, awn, acm, acdm, acb, afb in [
    (0.20, 0.30, 0.15, 0.08, 0.12, 0.10),
    (0.25, 0.35, 0.18, 0.10, 0.14, 0.12),
    (0.30, 0.40, 0.20, 0.10, 0.15, 0.12),
    (0.30, 0.35, 0.20, 0.12, 0.15, 0.12),
    (0.35, 0.40, 0.22, 0.12, 0.18, 0.14),
    (0.35, 0.45, 0.25, 0.12, 0.18, 0.14),
]:
    c = copy.deepcopy(ZERO_CFG)
    c['alpha'] = {'ST': ast,'Winger': awn,'CM': acm,'CDM': acdm,'CB': acb,'FB': afb}
    s = summary(sub, c)
    print(f"{ast:>6.2f} {awn:>6.2f} {acm:>5.2f} {acdm:>6.2f} {acb:>5.2f} {afb:>5.2f}  "
          f"{s.loc['ST','sc_gap']:>10.3f} {s.loc['Winger','sc_gap']:>11.3f} "
          f"{s.loc['CM','sc_gap']:>10.3f}  "
          f"{s.loc['ST','mean']:>8.3f} {s.loc['Winger','mean']:>9.3f} "
          f"{s.loc['CM','mean']:>8.3f} {s.loc['CB','mean']:>8.3f}")

Goal bonus sweep (α only):
  α_ST  α_Wng  α_CM  α_CDM  α_CB  α_FB   ST_sc_gap  Wng_sc_gap  CM_sc_gap   ST_mean  Wng_mean  CM_mean  CB_mean
--------------------------------------------------------------------------------------------------------------
  0.20   0.30  0.15   0.08  0.12  0.10       1.041       0.683      0.743     6.449     6.238    6.174    6.188
  0.25   0.35  0.18   0.10  0.14  0.12       1.102       0.744      0.778     6.475     6.253    6.181    6.189
  0.30   0.40  0.20   0.10  0.15  0.12       1.164       0.805      0.800     6.500     6.267    6.186    6.189
  0.30   0.35  0.20   0.12  0.15  0.12       1.164       0.744      0.800     6.500     6.253    6.186    6.189
  0.35   0.40  0.22   0.12  0.18  0.14       1.226       0.805      0.823     6.525     6.267    6.191    6.189
  0.35   0.45  0.25   0.12  0.18  0.14       1.226       0.866      0.857     6.525     6.282    6.198    6.189


In [7]:
# ── Sweep 2: Assist bonus (true contribution) ─────────────────────────────────
# bonus_contribution = rating_with_gamma - rating_without_gamma for assisters only.

ALPHA_FINAL = {'ST': 0.30, 'Winger': 0.40, 'CM': 0.20, 'CDM': 0.15, 'CB': 0.15, 'FB': 0.15}  # <-- update

assisters = sub[sub['assisted']].copy()
print(f"Assisters by group: {assisters.groupby('group').size().to_dict()}")
print()
print(f"{'γ_ST':>6} {'γ_Wng':>6} {'γ_CM':>5} {'γ_CDM':>6} {'γ_CB':>5} {'γ_FB':>5}  "
      f"{'ST_bonus':>9} {'Wng_bonus':>10} {'CM_bonus':>9}")
print("-" * 75)

for gst, gwn, gcm, gcdm, gcb, gfb in [
    (0.15, 0.15, 0.15, 0.10, 0.12, 0.12),
    (0.20, 0.20, 0.20, 0.12, 0.15, 0.15),
    (0.25, 0.25, 0.25, 0.15, 0.18, 0.18),
    (0.25, 0.30, 0.25, 0.15, 0.18, 0.18),
    (0.30, 0.30, 0.25, 0.15, 0.20, 0.20),
]:
    c_with = copy.deepcopy(ZERO_CFG)
    c_with['alpha'] = ALPHA_FINAL
    c_with['gamma'] = {'ST': gst,'Winger': gwn,'CM': gcm,'CDM': gcdm,'CB': gcb,'FB': gfb}
    c_zero = copy.deepcopy(c_with); c_zero['gamma'] = {}
    with_b = assisters.apply(lambda r: predict_rating(r, c_with), axis=1)
    zero_b = assisters.apply(lambda r: predict_rating(r, c_zero), axis=1)
    diff   = with_b - zero_b
    row_out = []
    for g in ['ST','Winger','CM']:
        idx = assisters[assisters['group']==g].index
        row_out.append(round(diff[idx].mean(), 3) if len(idx)>=3 else float('nan'))
    print(f"{gst:>6.2f} {gwn:>6.2f} {gcm:>5.2f} {gcdm:>6.2f} {gcb:>5.2f} {gfb:>5.2f}  "
          f"{row_out[0]:>9.3f} {row_out[1]:>10.3f} {row_out[2]:>9.3f}")

Assisters by group: {'CB': 11, 'CDM': 27, 'CM': 116, 'FB': 32, 'ST': 46, 'Winger': 113}

  γ_ST  γ_Wng  γ_CM  γ_CDM  γ_CB  γ_FB   ST_bonus  Wng_bonus  CM_bonus
---------------------------------------------------------------------------
  0.15   0.15  0.15   0.10  0.12  0.12      0.167      0.169     0.167
  0.20   0.20  0.20   0.12  0.15  0.15      0.223      0.226     0.222
  0.25   0.25  0.25   0.15  0.18  0.18      0.279      0.282     0.278
  0.25   0.30  0.25   0.15  0.18  0.18      0.279      0.339     0.278
  0.30   0.30  0.25   0.15  0.20  0.20      0.334      0.339     0.278


In [8]:
# ── Sweep 3: Mastery (proportional weight × excess cap) ───────────────────────
# Max bonus per condition per full game = excess_cap × weight.

GAMMA_FINAL = {'ST': 0.25, 'Winger': 0.25, 'CM': 0.25, 'CDM': 0.15, 'CB': 0.15, 'FB': 0.15}  # <-- update

MASTERY_WEIGHTS = [0.05, 0.08, 0.10, 0.12, 0.15]
EXCESS_CAPS     = [0.5, 1.0, 1.5, 2.0]

for cap in EXCESS_CAPS:
    print(f"\nexcess_cap={cap}  (max bonus/condition/full-game = {cap*0.15:.3f} at weight=0.15)")
    print(f"{'weight':>8}  " + "  ".join(f"{g+'_mean':>10}" for g in GROUPS))
    print("-" * 78)
    for w in MASTERY_WEIGHTS:
        c = copy.deepcopy(ZERO_CFG)
        c['alpha'] = ALPHA_FINAL; c['gamma'] = GAMMA_FINAL
        c['mastery_weight'] = w; c['mastery_excess_cap'] = cap
        s = summary(sub, c)
        means = "  ".join(f"{s.loc[g,'mean']:>10.3f}" if g in s.index else f"{'N/A':>10}" for g in GROUPS)
        print(f"{w:>8.2f}  {means}")


excess_cap=0.5  (max bonus/condition/full-game = 0.075 at weight=0.15)
  weight     ST_mean  Winger_mean     CM_mean    CDM_mean     CB_mean     FB_mean
------------------------------------------------------------------------------
    0.05       6.550       6.338       6.255       6.254       6.193       6.219
    0.08       6.551       6.339       6.256       6.255       6.194       6.220
    0.10       6.551       6.339       6.256       6.255       6.194       6.221
    0.12       6.552       6.340       6.257       6.256       6.194       6.221
    0.15       6.553       6.340       6.257       6.256       6.194       6.223

excess_cap=1.0  (max bonus/condition/full-game = 0.150 at weight=0.15)
  weight     ST_mean  Winger_mean     CM_mean    CDM_mean     CB_mean     FB_mean
------------------------------------------------------------------------------
    0.05       6.551       6.338       6.256       6.255       6.193       6.219
    0.08       6.553       6.339       6.256    

In [9]:
# ── Sweep 4: CDM Reliable Pivot ───────────────────────────────────────────────

MASTERY_WEIGHT_FINAL = 0.15   # <-- update
MASTERY_CAP_FINAL    = 2.0    # <-- update

c_base = copy.deepcopy(ZERO_CFG)
c_base.update({'alpha': ALPHA_FINAL, 'gamma': GAMMA_FINAL,
               'mastery_weight': MASTERY_WEIGHT_FINAL, 'mastery_excess_cap': MASTERY_CAP_FINAL})

cdm_base_mean = sub[sub['group']=='CDM'].apply(lambda r: predict_rating(r, c_base), axis=1).mean()
print(f"CDM mean pre-pivot (alpha+gamma+mastery): {cdm_base_mean:.3f}")
print()
print(f"{'PM_bonus':>9} {'RS_bonus':>9}  {'CDM_mean':>9} {'Uplift':>8}")
print("-" * 42)

for pm in [0.10, 0.15, 0.20, 0.25, 0.30]:
    for rs in [0.05, 0.08, 0.10, 0.12, 0.15]:
        c = copy.deepcopy(c_base)
        c['cdm_perfect_metronome'] = pm; c['cdm_reliable_shift'] = rs
        mean = sub[sub['group']=='CDM'].apply(lambda r: predict_rating(r, c), axis=1).mean()
        print(f"{pm:>9.2f} {rs:>9.2f}  {mean:>9.3f} {mean-cdm_base_mean:>+8.3f}")

CDM mean pre-pivot (alpha+gamma+mastery): 6.257

 PM_bonus  RS_bonus   CDM_mean   Uplift
------------------------------------------
     0.10      0.05      6.259   +0.002
     0.10      0.08      6.261   +0.004
     0.10      0.10      6.261   +0.004
     0.10      0.12      6.262   +0.005
     0.10      0.15      6.263   +0.006
     0.15      0.05      6.260   +0.003
     0.15      0.08      6.261   +0.004
     0.15      0.10      6.262   +0.005
     0.15      0.12      6.262   +0.005
     0.15      0.15      6.264   +0.007
     0.20      0.05      6.260   +0.003
     0.20      0.08      6.261   +0.004
     0.20      0.10      6.262   +0.005
     0.20      0.12      6.263   +0.006
     0.20      0.15      6.264   +0.007
     0.25      0.05      6.260   +0.003
     0.25      0.08      6.261   +0.004
     0.25      0.10      6.262   +0.005
     0.25      0.12      6.263   +0.006
     0.25      0.15      6.264   +0.007
     0.30      0.05      6.260   +0.003
     0.30      0.08      6.2

In [10]:
# ── Sweep 5: ST penalties and hold-up bonus ───────────────────────────────────

CDM_PM_FINAL = 0.30   # <-- update
CDM_RS_FINAL = 0.15   # <-- update

c_st_base = copy.deepcopy(c_base)
c_st_base.update({'cdm_perfect_metronome': CDM_PM_FINAL, 'cdm_reliable_shift': CDM_RS_FINAL})

st_sub = sub[sub['group']=='ST'].copy()
st_base_mean = st_sub.apply(lambda r: predict_rating(r, c_st_base), axis=1).mean()
print(f"ST mean before penalties/hold-up: {st_base_mean:.3f}")
print()

for label, param_pairs, keys in [
    ('Hold-Up Bonus (scale, cap)',
     [(0.008,0.20),(0.010,0.25),(0.012,0.30),(0.015,0.30),(0.015,0.35)],
     ('st_holdup_scale','st_holdup_cap')),
    ('Black Hole Penalty (scale, cap)',
     [(0.03,0.25),(0.04,0.30),(0.04,0.40),(0.05,0.40),(0.06,0.50)],
     ('st_blackhole_scale','st_blackhole_cap')),
    ('Wasteful Finisher Penalty (scale, cap)',
     [(0.08,0.30),(0.10,0.35),(0.12,0.40),(0.15,0.45),(0.18,0.50)],
     ('st_wasteful_scale','st_wasteful_cap')),
]:
    print(f"{label}:")
    print(f"  {'scale':>8} {'cap':>6}  {'ST_mean':>8} {'Delta':>8}")
    for s, cap in param_pairs:
        c = copy.deepcopy(c_st_base)
        c[keys[0]] = s; c[keys[1]] = cap
        mean = st_sub.apply(lambda r: predict_rating(r, c), axis=1).mean()
        print(f"  {s:>8.3f} {cap:>6.2f}  {mean:>8.3f} {mean-st_base_mean:>+8.3f}")
    print()

print("Combined ST mechanics sweep:")
print(f"  {'hu_s':>5} {'hu_c':>5} {'bh_s':>5} {'bh_c':>5} {'wf_s':>5} {'wf_c':>5}  {'ST_mean':>8} {'Delta':>8}")
print("  " + "-" * 65)
for hu_s,hu_c,bh_s,bh_c,wf_s,wf_c in [
    (0.010,0.25,0.040,0.35,0.12,0.40),
    (0.010,0.25,0.040,0.40,0.15,0.45),
    (0.012,0.30,0.050,0.40,0.15,0.45),
    (0.012,0.30,0.050,0.45,0.18,0.50),
    (0.015,0.30,0.050,0.45,0.18,0.50),
]:
    c = copy.deepcopy(c_st_base)
    c.update({'st_holdup_scale':hu_s,'st_holdup_cap':hu_c,
              'st_blackhole_scale':bh_s,'st_blackhole_cap':bh_c,
              'st_wasteful_scale':wf_s,'st_wasteful_cap':wf_c})
    mean = st_sub.apply(lambda r: predict_rating(r, c), axis=1).mean()
    print(f"  {hu_s:>5.3f} {hu_c:>5.2f} {bh_s:>5.3f} {bh_c:>5.2f} {wf_s:>5.2f} {wf_c:>5.2f}  "
          f"{mean:>8.3f} {mean-st_base_mean:>+8.3f}")

ST mean before penalties/hold-up: 6.560

Hold-Up Bonus (scale, cap):
     scale    cap   ST_mean    Delta
     0.008   0.20     6.591   +0.030
     0.010   0.25     6.598   +0.038
     0.012   0.30     6.606   +0.045
     0.015   0.30     6.607   +0.047
     0.015   0.35     6.614   +0.054

Black Hole Penalty (scale, cap):
     scale    cap   ST_mean    Delta
     0.030   0.25     6.560   +0.000
     0.040   0.30     6.560   +0.000
     0.040   0.40     6.560   +0.000
     0.050   0.40     6.560   +0.000
     0.060   0.50     6.560   +0.000

Wasteful Finisher Penalty (scale, cap):
     scale    cap   ST_mean    Delta
     0.080   0.30     6.559   -0.002
     0.100   0.35     6.559   -0.002
     0.120   0.40     6.558   -0.002
     0.150   0.45     6.558   -0.003
     0.180   0.50     6.557   -0.003

Combined ST mechanics sweep:
   hu_s  hu_c  bh_s  bh_c  wf_s  wf_c   ST_mean    Delta
  -----------------------------------------------------------------
  0.010  0.25 0.040  0.35  0.12  0.

In [11]:
# ── Sweep 6: CB/FB xG-tiered clean sheet + collapse penalty ───────────────────

ST_HOLDUP_SCALE_FINAL    = 0.012;  ST_HOLDUP_CAP_FINAL    = 0.30   # <-- update
ST_BLACKHOLE_SCALE_FINAL = 0.060;  ST_BLACKHOLE_CAP_FINAL = 0.50   # <-- update
ST_WASTEFUL_SCALE_FINAL  = 0.180;  ST_WASTEFUL_CAP_FINAL  = 0.50   # <-- update

c_def_base = copy.deepcopy(c_st_base)
c_def_base.update({
    'st_holdup_scale':    ST_HOLDUP_SCALE_FINAL,   'st_holdup_cap':    ST_HOLDUP_CAP_FINAL,
    'st_blackhole_scale': ST_BLACKHOLE_SCALE_FINAL, 'st_blackhole_cap': ST_BLACKHOLE_CAP_FINAL,
    'st_wasteful_scale':  ST_WASTEFUL_SCALE_FINAL,  'st_wasteful_cap':  ST_WASTEFUL_CAP_FINAL,
})

for g in ['CB','FB']:
    bm = sub[sub['group']==g].apply(lambda r: predict_rating(r, c_def_base), axis=1).mean()
    print(f"{g} pre-CS/collapse baseline: {bm:.3f}")
print()

print(f"{'low_xg':>7} {'mid_xg':>7} {'high_xg':>8} {'fb_r':>6} {'coll':>6}  {'CB_mean':>8} {'FB_mean':>8}")
print("-" * 65)

for low,mid,high in [(0.25,0.16,0.06),(0.28,0.18,0.07),(0.30,0.20,0.08),(0.33,0.22,0.09),(0.35,0.24,0.10),(0.40,0.28,0.12),(0.45,0.32,0.14),(0.50,0.36,0.16)]:
    for fb_r in [0.60, 0.70, 0.75]:
        for col in [0.15, 0.20, 0.25]:
            c = copy.deepcopy(c_def_base)
            c.update({'cs_cb_low_xg':low,'cs_cb_mid_xg':mid,'cs_cb_high_xg':high,
                      'cs_fb_ratio':fb_r,'collapse_penalty':col})
            cb_m = sub[sub['group']=='CB'].apply(lambda r: predict_rating(r, c), axis=1).mean()
            fb_m = sub[sub['group']=='FB'].apply(lambda r: predict_rating(r, c), axis=1).mean()
            print(f"{low:>7.2f} {mid:>7.2f} {high:>8.2f} {fb_r:>6.2f} {col:>6.2f}  {cb_m:>8.3f} {fb_m:>8.3f}")

CB pre-CS/collapse baseline: 6.195
FB pre-CS/collapse baseline: 6.224

 low_xg  mid_xg  high_xg   fb_r   coll   CB_mean  FB_mean
-----------------------------------------------------------------
   0.25    0.16     0.06   0.60   0.15     6.288    6.280
   0.25    0.16     0.06   0.60   0.20     6.286    6.278
   0.25    0.16     0.06   0.60   0.25     6.284    6.276
   0.25    0.16     0.06   0.70   0.15     6.288    6.290
   0.25    0.16     0.06   0.70   0.20     6.286    6.288
   0.25    0.16     0.06   0.70   0.25     6.284    6.287
   0.25    0.16     0.06   0.75   0.15     6.288    6.295
   0.25    0.16     0.06   0.75   0.20     6.286    6.293
   0.25    0.16     0.06   0.75   0.25     6.284    6.292
   0.28    0.18     0.07   0.60   0.15     6.300    6.287
   0.28    0.18     0.07   0.60   0.20     6.298    6.286
   0.28    0.18     0.07   0.60   0.25     6.296    6.284
   0.28    0.18     0.07   0.70   0.15     6.300    6.299
   0.28    0.18     0.07   0.70   0.20     6.298   

In [12]:
# ── Sweep 7: CDM/CM xG-tiered clean sheet ────────────────────────────────────
# CDM/CM now use the same CB xG tiers scaled by a position ratio.
#   cs_cdm_ratio: CDM gets this fraction of CB's tier values
#   cs_cm_ratio:  CM gets this fraction of CB's tier values
# Effective absolute values shown for reference.

CS_CB_LOW_FINAL   = 0.50;  CS_CB_MID_FINAL  = 0.36
CS_CB_HIGH_FINAL  = 0.16;  CS_FB_RATIO_FINAL = 0.60
COLLAPSE_FINAL    = 0.20   # <-- update all above

c_mid_base = copy.deepcopy(c_def_base)
c_mid_base.update({
    'cs_cb_low_xg': CS_CB_LOW_FINAL,  'cs_cb_mid_xg': CS_CB_MID_FINAL,
    'cs_cb_high_xg': CS_CB_HIGH_FINAL, 'cs_fb_ratio': CS_FB_RATIO_FINAL,
    'collapse_penalty': COLLAPSE_FINAL,
    'cs_cdm_ratio': 0.0, 'cs_cm_ratio': 0.0,
})

for g in ['CDM', 'CM']:
    bm = sub[sub['group']==g].apply(lambda r: predict_rating(r, c_mid_base), axis=1).mean()
    print(f"{g} pre-CS baseline: {bm:.3f}")
print()

print(f"{'cdm_r':>7} {'cm_r':>6}  "
      f"{'CDM_low':>8} {'CDM_mid':>8} {'CDM_high':>9}  "
      f"{'CM_low':>7} {'CM_mid':>7} {'CM_high':>8}  "
      f"{'CDM_mean':>9} {'CM_mean':>8}")
print("-" * 95)

for cdm_r in [0.28, 0.32, 0.36, 0.40, 0.44, 0.48]:
    for cm_r in [0.20, 0.24, 0.28, 0.32, 0.36, 0.40]:
        c = copy.deepcopy(c_mid_base)
        c.update({'cs_cdm_ratio': cdm_r, 'cs_cm_ratio': cm_r})
        cdm_m = sub[sub['group']=='CDM'].apply(lambda r: predict_rating(r, c), axis=1).mean()
        cm_m  = sub[sub['group']=='CM' ].apply(lambda r: predict_rating(r, c), axis=1).mean()
        cdm_low  = round(CS_CB_LOW_FINAL  * cdm_r, 3)
        cdm_mid  = round(CS_CB_MID_FINAL  * cdm_r, 3)
        cdm_high = round(CS_CB_HIGH_FINAL * cdm_r, 3)
        cm_low   = round(CS_CB_LOW_FINAL  * cm_r,  3)
        cm_mid   = round(CS_CB_MID_FINAL  * cm_r,  3)
        cm_high  = round(CS_CB_HIGH_FINAL * cm_r,  3)
        print(f"{cdm_r:>7.2f} {cm_r:>6.2f}  "
              f"{cdm_low:>8.3f} {cdm_mid:>8.3f} {cdm_high:>9.3f}  "
              f"{cm_low:>7.3f} {cm_mid:>7.3f} {cm_high:>8.3f}  "
              f"{cdm_m:>9.3f} {cm_m:>8.3f}")


CDM pre-CS baseline: 6.264
CM pre-CS baseline: 6.258

  cdm_r   cm_r   CDM_low  CDM_mid  CDM_high   CM_low  CM_mid  CM_high   CDM_mean  CM_mean
-----------------------------------------------------------------------------------------------
   0.28   0.20     0.140    0.101     0.045    0.100   0.072    0.032      6.319    6.299
   0.28   0.24     0.140    0.101     0.045    0.120   0.086    0.038      6.319    6.307
   0.28   0.28     0.140    0.101     0.045    0.140   0.101    0.045      6.319    6.315
   0.28   0.32     0.140    0.101     0.045    0.160   0.115    0.051      6.319    6.323
   0.28   0.36     0.140    0.101     0.045    0.180   0.130    0.058      6.319    6.331
   0.28   0.40     0.140    0.101     0.045    0.200   0.144    0.064      6.319    6.339
   0.32   0.20     0.160    0.115     0.051    0.100   0.072    0.032      6.327    6.299
   0.32   0.24     0.160    0.115     0.051    0.120   0.086    0.038      6.327    6.307
   0.32   0.28     0.160    0.115     0.

In [13]:
# ── Sweep 8: Winger Wastefulness Penalty ─────────────────────────────────────
# Fires when wasted_shots (shots - goals) >= 3 AND creative output (goals+assists) == 0.
# Penalty = (wasted_shots - 2) × winger_wasteful_per_shot
# Fix cs_cdm_ratio and cs_cm_ratio before running.

CS_CDM_RATIO_FINAL = 0.40  # <-- update from sweep 7
CS_CM_RATIO_FINAL  = 0.28  # <-- update from sweep 7

c_wng_base = copy.deepcopy(c_mid_base)
c_wng_base.update({'cs_cdm_ratio': CS_CDM_RATIO_FINAL, 'cs_cm_ratio': CS_CM_RATIO_FINAL,
                   'winger_wasteful_per_shot': 0.0})

# Diagnostic: trigger rates at current thresholds
wng = sub[sub['group'] == 'Winger'].copy()
wng['wasted']   = wng['shots_raw'] - wng['goals']
wng['creative'] = wng['goals'] + wng['assisted'].astype(int)
triggered = wng[(wng['wasted'] >= 3) & (wng['creative'] == 0)]
print(f"Trigger rate: {len(triggered)} / {len(wng)} ({len(triggered)/len(wng):.1%})")
print(f"Mean wasted shots when triggered: {triggered['wasted'].mean():.2f}")
print(f"Max wasted shots: {triggered['wasted'].max():.0f}")
print()

wng_base_mean = wng.apply(lambda r: predict_rating(r, c_wng_base), axis=1).mean()
print(f"Winger mean before penalty: {wng_base_mean:.3f}")
print()
print(f"{'per_shot':>9}  {'Wng_mean':>9} {'Delta':>8} {'mean_penalty_triggered':>23}")
print("-" * 55)

for ps in [0.02, 0.03, 0.05, 0.07, 0.10, 0.12, 0.15]:
    c = copy.deepcopy(c_wng_base)
    c['winger_wasteful_per_shot'] = ps
    preds = wng.apply(lambda r: predict_rating(r, c), axis=1)
    mean  = preds.mean()
    # Mean penalty on triggered appearances only
    base_preds  = wng.apply(lambda r: predict_rating(r, c_wng_base), axis=1)
    mean_pen    = (base_preds[triggered.index] - preds[triggered.index]).mean()
    print(f"{ps:>9.2f}  {mean:>9.3f} {mean-wng_base_mean:>+8.3f} {mean_pen:>23.3f}")


Trigger rate: 21 / 459 (4.6%)
Mean wasted shots when triggered: 3.52
Max wasted shots: 5

Winger mean before penalty: 6.342

 per_shot   Wng_mean    Delta  mean_penalty_triggered
-------------------------------------------------------
     0.02      6.341   -0.001                   0.030
     0.03      6.340   -0.002                   0.046
     0.05      6.338   -0.003                   0.076
     0.07      6.337   -0.005                   0.107
     0.10      6.335   -0.007                   0.152
     0.12      6.334   -0.008                   0.183
     0.15      6.332   -0.010                   0.229


In [14]:
# ── Final configuration ───────────────────────────────────────────────────────
# Update every value below after completing the sweeps above.

FINAL_CFG = {
    'alpha': {'ST': 0.30, 'Winger': 0.40, 'CM': 0.20, 'CDM': 0.15, 'CB': 0.15, 'FB': 0.15},
    'gamma': {'ST': 0.25, 'Winger': 0.25, 'CM': 0.25, 'CDM': 0.15, 'CB': 0.15, 'FB': 0.15},
    'mastery_weight': 0.15,    'mastery_excess_cap': 2.0,
    'cdm_perfect_metronome': 0.30,  'cdm_reliable_shift': 0.15,
    'st_holdup_scale': 0.012,  'st_holdup_cap': 0.30,
    'st_blackhole_scale': 0.060, 'st_blackhole_cap': 0.50,
    'st_wasteful_scale': 0.18, 'st_wasteful_cap': 0.50,
    'winger_wasteful_per_shot': 0.15,
    'cs_cb_low_xg': 0.50,  'cs_cb_mid_xg': 0.36,  'cs_cb_high_xg': 0.16,
    'cs_fb_ratio': 0.60,   'cs_cdm_ratio': 0.48,  'cs_cm_ratio': 0.36,
    'cs_ramp_n': 1.0,      'collapse_penalty': 0.20,
}

print("FINAL CONFIGURATION")
for k, v in FINAL_CFG.items():
    print(f"  {k:<30} {v}")
print()

final_s = summary(sub, FINAL_CFG)
print(final_s.to_string())
print()

all_pred = sub.apply(lambda r: predict_rating(r, FINAL_CFG), axis=1)
print(f"Overall outfield mean:   {all_pred.mean():.3f}")
print(f"Overall outfield median: {all_pred.median():.3f}")
print(f"Uplift from base_rating: {all_pred.mean() - sub['base_rating'].mean():+.3f}")

FINAL CONFIGURATION
  alpha                          {'ST': 0.3, 'Winger': 0.4, 'CM': 0.2, 'CDM': 0.15, 'CB': 0.15, 'FB': 0.15}
  gamma                          {'ST': 0.25, 'Winger': 0.25, 'CM': 0.25, 'CDM': 0.15, 'CB': 0.15, 'FB': 0.15}
  mastery_weight                 0.15
  mastery_excess_cap             2.0
  cdm_perfect_metronome          0.3
  cdm_reliable_shift             0.15
  st_holdup_scale                0.012
  st_holdup_cap                  0.3
  st_blackhole_scale             0.06
  st_blackhole_cap               0.5
  st_wasteful_scale              0.18
  st_wasteful_cap                0.5
  winger_wasteful_per_shot       0.15
  cs_cb_low_xg                   0.5
  cs_cb_mid_xg                   0.36
  cs_cb_high_xg                  0.16
  cs_fb_ratio                    0.6
  cs_cdm_ratio                   0.48
  cs_cm_ratio                    0.36
  cs_ramp_n                      1.0
  collapse_penalty               0.2

          n   mean    p25    p50    p75  non_s

In [15]:
import pprint
print("Constants for Phase 8 service patch:")
print()
pprint.pprint(FINAL_CFG)

Constants for Phase 8 service patch:

{'alpha': {'CB': 0.15,
           'CDM': 0.15,
           'CM': 0.2,
           'FB': 0.15,
           'ST': 0.3,
           'Winger': 0.4},
 'cdm_perfect_metronome': 0.3,
 'cdm_reliable_shift': 0.15,
 'collapse_penalty': 0.2,
 'cs_cb_high_xg': 0.16,
 'cs_cb_low_xg': 0.5,
 'cs_cb_mid_xg': 0.36,
 'cs_cdm_ratio': 0.48,
 'cs_cm_ratio': 0.36,
 'cs_fb_ratio': 0.6,
 'cs_ramp_n': 1.0,
 'gamma': {'CB': 0.15,
           'CDM': 0.15,
           'CM': 0.25,
           'FB': 0.15,
           'ST': 0.25,
           'Winger': 0.25},
 'mastery_excess_cap': 2.0,
 'mastery_weight': 0.15,
 'st_blackhole_cap': 0.5,
 'st_blackhole_scale': 0.06,
 'st_holdup_cap': 0.3,
 'st_holdup_scale': 0.012,
 'st_wasteful_cap': 0.5,
 'st_wasteful_scale': 0.18,
 'winger_wasteful_per_shot': 0.15}
